In [1]:
%pip install "qiskit==2.2.3" "qiskit-machine-learning==0.9.0" "qiskit-optimization==0.7.0" "qiskit-algorithms==0.4.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.0/664.0 kB 20.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 54.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 54.4 MB/s  0:00:00
  Created wheel for docplex: filename=docplex-2.32.264-py3-none-any.whl size=703859 sha256=ec5f674bcf84d68d61b01cd3379caf1ddb984603e90ce9b06558131a8193dc9d
  Stored in directory: /root/.cache/pip/wheels/ba/11/de/5d0051be010658975fb2155006b4b8a818136d9f69ed20b4ba
Successfully built docplex
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [qiskit-algorithms]


In [9]:
import sys
import importlib.metadata as metadata
import numpy as np
import scipy
import sklearn
import qiskit
import qiskit_machine_learning
import qiskit_optimization
import qiskit_algorithms

expected = {
    "qiskit": "2.2.3",
    "qiskit-machine-learning": "0.9.0",
    "qiskit-optimization": "0.7.0",
    "qiskit-algorithms": "0.4.0",
}
print("Python:", sys.version.split()[0])
for package in [*expected, "numpy", "scipy", "scikit-learn"]:
    installed = metadata.version(package)
    print(f"{package}: {installed}")
    if package in expected:
        assert installed == expected[package], (
            f"{package} 버전을 확인하고 설치 셀 실행 후 세션을 다시 시작하세요."
        )
print("패키지 import와 버전 확인 완료")

Python: 3.13.18
qiskit: 2.2.3
qiskit-machine-learning: 0.9.0
qiskit-optimization: 0.7.0
qiskit-algorithms: 0.4.0
numpy: 2.4.6
scipy: 1.16.3
scikit-learn: 1.6.1
패키지 import와 버전 확인 완료


In [5]:
import numpy as np

from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit.library import n_local
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import COBYLA



# -----------------------------------------
# 1. 해밀토니안 (보안 정책 위험도)
# -----------------------------------------
paulis = [
    ("ZII", 0.8),
    ("IZI", 0.6),
    ("IIZ", 0.4),
    ("ZZI", 0.5),
    ("ZIZ", 0.3),
    ("IZZ", 0.2),
]

H = SparsePauliOp.from_list(paulis)

# -----------------------------------------
# 2. Ansatz
# -----------------------------------------
ansatz = n_local(
    num_qubits=3,
    reps=2,
    rotation_blocks="ry",
    entanglement_blocks="cz",
    entanglement="full"
)

# -----------------------------------------
# 3. VQE 실행
# -----------------------------------------
estimator = StatevectorEstimator()
optimizer = COBYLA(maxiter=100)

vqe = VQE(estimator=estimator, ansatz=ansatz, optimizer=optimizer,
          initial_point=np.linspace(0.1, 1.0, ansatz.num_parameters))

vqe_result = vqe.compute_minimum_eigenvalue(H)

print("VQE 최소 에너지(위험도):", vqe_result.eigenvalue.real)

# -----------------------------------------
# 4. 최적 파라미터로 Statevector 직접 생성
# -----------------------------------------
params = vqe_result.optimal_point
sv = Statevector(ansatz.assign_parameters(params))

# 확률 분포 계산
probs = sv.probabilities_dict()   # {bitstring: prob}

# 최적 비트스트링 선택
best_bitstring = max(probs, key=probs.get)

# Qiskit 비트순서 보정
best_bitstring_fixed = best_bitstring[::-1]

print("\n최적 비트스트링:", best_bitstring_fixed)
print("확률:", probs[best_bitstring])

# -----------------------------------------
# 5. 보안 정책 해석
# -----------------------------------------
policy_names = ["방화벽", "접근 통제", "로그 정책"]

print("\n=== 최적 보안 설정 조합 ===")
for i, bit in enumerate(best_bitstring_fixed):
    status = "ON(활성)" if bit == "1" else "OFF(비활성)"
    print(f"{policy_names[i]}: {status}")

energies = np.real(np.diag(H.to_matrix()))
exact = float(energies.min())
chosen_energy = float(energies[int(best_bitstring, 2)])
print("정확 최소 에너지:", exact)
print("선택 구성 에너지:", chosen_energy)
print("선택 구성과 정확 최적값 차이:", chosen_energy - exact)
print("기대 에너지와 정확 최적값 차이:", float(vqe_result.eigenvalue.real) - exact)
assert np.isclose(sum(probs.values()), 1.0)
assert vqe_result.eigenvalue.real >= exact - 1e-8

x: [1 0 1 0] 슬랙: 0 원래 비용: -12.5 QUBO: -12.5 z: [-1  1 -1  1]
x: [1 0 0 0] 슬랙: 1 원래 비용: -6.5 QUBO: -6.5 z: [-1  1  1  1]


정책 변수는 q0=방화벽, q1=접근 통제, q2=로그 정책입니다. Qiskit 파울리 문자열의 오른쪽 문자가 q0에 대응하므로 ZII의 0.8은 로그 정책, IZI의 0.6은 접근 통제, IIZ의 0.4는 방화벽 계수입니다. 계산 기준은 z_i=1-2x_i이며 비활성 x_i=0은 z_i=+1, 활성 x_i=1은 z_i=-1에 대응합니다. 양의 ZZ 계수는 두 비트가 같을 때 양의 에너지, 다를 때 음의 에너지를 더합니다.

n_local은 RY 회전과 CZ 얽힘을 반복하는 안자츠를 구성합니다. θ(세타)는 각 RY 회전의 각도이며 COBYLA가 θ(세타)를 조정합니다. StatevectorEstimator는 각 상태의 확률에 에너지를 곱하여 합한 기대 에너지를 계산합니다. 최적화 후 Statevector로 확률을 계산하고 가장 높은 확률의 구성을 출력합니다.

실행 예의 기대 에너지는 약 -0.999717이고 정확 최소 에너지는 -1.0입니다. 표시된 정책 비트 [방화벽, 접근 통제, 로그 정책]는 011이며 해당 구성의 에너지는 -1.0, 확률은 약 0.998055입니다. 확률이 가장 높은 구성의 에너지와 기대 에너지는 각각 출력됩니다. 근사 최적화의 결과는 초기 각도와 반복 횟수에 따라 달라질 수 있습니다.

이 해밀토니안은 단위가 없는 교육용 평가 함수입니다. 방화벽 비활성 결과는 주어진 계수의 수학적 해석입니다. 실제 운영 정책에는 방화벽 필수 활성과 같은 제약 및 실측 위험도를 반영해야 합니다.

###